In [3]:
import pandas as pd
import re
import random

In [4]:
def listToLabel(head_labels):
    ct = {}
    for i in range(len(head_labels)):
        ct['column' + str(i+1)] = head_labels[i]
    return ct

In [5]:
head_labels = 'Year,Quarter,Month,DayofMonth,DayOfWeek,FlightDate,UniqueCarrier,AirlineID,Carrier,TailNum,FlightNum,OriginAirportID,OriginAirportSeqID,OriginCityMarketID,Origin,OriginCityName,OriginState,OriginStateFips,OriginStateName,OriginWac,DestAirportID,DestAirportSeqID,DestCityMarketID,Dest,DestCityName,DestState,DestStateFips,DestStateName,DestWac,CRSDepTime,DepTime,DepDelay,DepDelayMinutes,DepDel15,DepartureDelayGroups,DepTimeBlk,TaxiOut,TaxiIn,CRSArrTime,ArrTime,ArrDelay,ArrDelayMinutes,ArrDel15,ArrivalDelayGroups,ArrTimeBlk,Cancelled,CancellationCode,Diverted,CRSElapsedTime,ActualElapsedTime,AirTime,Flights,Distance,DistanceGroup,CarrierDelay,WeatherDelay,NASDelay,SecurityDelay,LateAircraftDelay,FirstDepTime,TotalAddGTime,LongestAddGTime'.split(',')
columnNameToLabel = listToLabel(head_labels)

In [6]:
def lineToDataStruct(lineElements, columnNameToLabel):

    index_dependant = 0
    for element in lineElements:
        if element.startswith('"dependant":'):
            break
        index_dependant += 1

    rhs = lineElements[index_dependant + 1].split(':')[1]
    for removeChar in ['"','[',']','{','}']:
        rhs = rhs.replace(removeChar, '')
    determinant_index = 0
    rhs = columnNameToLabel[rhs]

    for element in lineElements:
        if element.startswith('"determinant'):
            #print('found at ' + str(determinant_index))
            break
        determinant_index += 1

    X = [] # RHS
    for s in lineElements[determinant_index + 1: index_dependant]:
        match = re.search(r'"columnIdentifier":"(column\d+)"', s)
        if match:
            result = match.group(1)
            X.append(columnNameToLabel[result])

    return X,rhs

In [7]:
data = []
with open('2025-11-14_17-00-10_fds.csv', 'r') as file:
    for line in file:
        lhs, rhs = lineToDataStruct(line.strip().split(","), columnNameToLabel)
        prettyPrint = lambda lst: ', '.join(lst)
        #print(prettyPrint(lhs), '->',rhs)
        data .append( (lhs, rhs) )

In [8]:
ids = random.sample(range(len(data)), 100)

In [9]:
sample = [data[i] for i in ids]

In [10]:
sample[2]

(['TailNum',
  'OriginCityName',
  'DestCityMarketID',
  'ArrDelayMinutes',
  'ArrTimeBlk',
  'ActualElapsedTime',
  'AirTime',
  'FirstDepTime'],
 'DepDel15')

In [11]:
def find_fds_for_rhs(target_rhs, data,sorted=False, lhs_validator = lambda x: True, lhs_size_min = -1, lhs_size_max = -1):
    results = []
    for tuple in data:
        lhs, rhs = tuple
        if rhs == target_rhs and lhs_validator(lhs):
            results.append(tuple)
    if sorted:
        results.sort(key=lambda x: len(x[0]))
    size_filter = lambda lhs: (lhs_size_min < 0 or len(lhs) >= lhs_size_min ) and (lhs_size_max < 0 or len(lhs) <= lhs_size_max)
    return [t for t in results if size_filter(t[0])]

In [12]:
genuine_fds = [find_fds_for_rhs('ArrDelay', data, sorted = True)[0]]

In [13]:
firsts = [find_fds_for_rhs(label, data, sorted = True)[0] for label in head_labels]

In [14]:
firsts

[(['ArrTime'], 'Year'),
 (['ArrTime'], 'Quarter'),
 (['ArrTime'], 'Month'),
 (['FlightDate'], 'DayofMonth'),
 (['DayofMonth'], 'DayOfWeek'),
 (['DayofMonth'], 'FlightDate'),
 (['ArrTime'], 'UniqueCarrier'),
 (['ArrTime'], 'AirlineID'),
 (['ArrTime'], 'Carrier'),
 (['FlightNum', 'CRSArrTime', 'DayofMonth'], 'TailNum'),
 (['CRSDepTime', 'CRSArrTime', 'DayofMonth'], 'FlightNum'),
 (['OriginAirportSeqID'], 'OriginAirportID'),
 (['OriginAirportID'], 'OriginAirportSeqID'),
 (['OriginAirportID'], 'OriginCityMarketID'),
 (['OriginAirportID'], 'Origin'),
 (['OriginAirportID'], 'OriginCityName'),
 (['OriginAirportID'], 'OriginState'),
 (['OriginAirportID'], 'OriginStateFips'),
 (['OriginAirportID'], 'OriginStateName'),
 (['OriginAirportID'], 'OriginWac'),
 (['DestAirportSeqID'], 'DestAirportID'),
 (['DestAirportID'], 'DestAirportSeqID'),
 (['DestAirportID'], 'DestCityMarketID'),
 (['DestAirportID'], 'Dest'),
 (['DestAirportID'], 'DestCityName'),
 (['DestAirportID'], 'DestState'),
 (['DestAirport

In [15]:
def create_sample(data, n):
    ids = random.sample(range(len(data)), n)
    sample = [data[i] for i in ids]
    return sample

In [16]:
import csv
import os

def export_sample_to_csv(filename):
    if not os.path.exists(filename):
        with open(filename, 'w', newline='') as f:
            writer = csv.writer(f)
            write_back = [['LHS', 'RHS']]
            write_back.extend(create_sample(data, 1000))
            writer.writerows(write_back)


In [17]:
export_sample_to_csv( 'training_data.csv')

In [18]:
find_fds_for_rhs('NASDelay', data, sorted = True)

[(['TailNum', 'DepTime', 'ArrTime'], 'NASDelay'),
 (['DepTime', 'CRSArrTime', 'ArrTime'], 'NASDelay'),
 (['DepTime', 'ArrTime', 'ArrDelay'], 'NASDelay'),
 (['DepTime', 'ArrTime', 'ArrDelayMinutes'], 'NASDelay'),
 (['DepTime', 'ArrTime', 'CarrierDelay'], 'NASDelay'),
 (['TailNum', 'ArrTime', 'ActualElapsedTime'], 'NASDelay'),
 (['TailNum', 'ArrTime', 'AirTime'], 'NASDelay'),
 (['TailNum', 'DepDelay', 'ArrTime'], 'NASDelay'),
 (['TailNum', 'ArrTime', 'CarrierDelay'], 'NASDelay'),
 (['TailNum', 'DayofMonth', 'ArrTime'], 'NASDelay'),
 (['TailNum', 'ArrTime', 'FlightDate'], 'NASDelay'),
 (['CRSArrTime', 'ArrTime', 'ActualElapsedTime'], 'NASDelay'),
 (['DepDelay', 'CRSArrTime', 'ArrTime'], 'NASDelay'),
 (['DepDelayMinutes', 'CRSArrTime', 'ArrTime'], 'NASDelay'),
 (['CRSArrTime', 'DayofMonth', 'ArrTime'], 'NASDelay'),
 (['CRSArrTime', 'ArrTime', 'FlightDate'], 'NASDelay'),
 (['ArrTime', 'ArrDelay', 'ActualElapsedTime'], 'NASDelay'),
 (['ArrTime', 'ArrDelayMinutes', 'ActualElapsedTime'], 'NASD

In [19]:
find_fds_for_rhs('FlightNum', data, sorted = True, lhs_validator = lambda lhs: any(['origin' in label.lower() for label in lhs]) and not any(['city' in label.lower() for label in lhs]))

[(['TailNum', 'OriginAirportID', 'CRSDepTime', 'CRSArrTime'], 'FlightNum'),
 (['TailNum', 'OriginAirportSeqID', 'CRSDepTime', 'CRSArrTime'], 'FlightNum'),
 (['TailNum', 'Origin', 'CRSDepTime', 'CRSArrTime'], 'FlightNum'),
 (['TailNum', 'OriginAirportID', 'CRSArrTime', 'CRSElapsedTime'], 'FlightNum'),
 (['TailNum', 'OriginAirportSeqID', 'CRSArrTime', 'CRSElapsedTime'],
  'FlightNum'),
 (['TailNum', 'Origin', 'CRSArrTime', 'CRSElapsedTime'], 'FlightNum'),
 (['TailNum', 'OriginAirportID', 'CRSArrTime', 'Distance'], 'FlightNum'),
 (['TailNum', 'OriginAirportSeqID', 'CRSArrTime', 'Distance'], 'FlightNum'),
 (['TailNum', 'Origin', 'CRSArrTime', 'Distance'], 'FlightNum'),
 (['TailNum', 'OriginState', 'CRSArrTime', 'Distance'], 'FlightNum'),
 (['TailNum', 'OriginStateFips', 'CRSArrTime', 'Distance'], 'FlightNum'),
 (['TailNum', 'OriginStateName', 'CRSArrTime', 'Distance'], 'FlightNum'),
 (['TailNum', 'OriginWac', 'CRSArrTime', 'Distance'], 'FlightNum'),
 (['TailNum', 'OriginAirportID', 'DestAi

In [20]:

def build_lhs_validator(required_substrings, forbidden_substrings):
    def lhs_validator(lhs):
        lhs_lower = [label.lower() for label in lhs]
        if ( len(required_substrings) > 0):
            for req in required_substrings:
                if not any([req in label for label in lhs_lower]):
                    return False
        else: 
            print('awdsad')
        for forb in forbidden_substrings:
            if any([forb in label for label in lhs_lower]):
                return False
        return True
    return lhs_validator

In [21]:
new_file_path = '2025-11-21_14-13-53_fds.csv'

In [22]:
new_labels = 'Car_id,Date,Customer Name,Gender,Annual Income,Dealer_Name,Company,Model,Engine,Transmission,Color,Price ($),Dealer_No ,Body Style,Phone,Dealer_Region'.split(',')

In [23]:
new_ct = listToLabel(new_labels)

In [24]:
new_ct

{'column1': 'Car_id',
 'column2': 'Date',
 'column3': 'Customer Name',
 'column4': 'Gender',
 'column5': 'Annual Income',
 'column6': 'Dealer_Name',
 'column7': 'Company',
 'column8': 'Model',
 'column9': 'Engine',
 'column10': 'Transmission',
 'column11': 'Color',
 'column12': 'Price ($)',
 'column13': 'Dealer_No ',
 'column14': 'Body Style',
 'column15': 'Phone',
 'column16': 'Dealer_Region'}

In [25]:
newData = []
with open(new_file_path, 'r') as file:
    for line in file:
        print(line)
        lhs, rhs = lineToDataStruct(line.strip().split(","), new_ct)
        prettyPrint = lambda lst: ', '.join(lst)
        print(prettyPrint(lhs), '->',rhs)
        newData.append( (lhs, rhs) )

{"type":"FunctionalDependency","determinant":{"columnIdentifiers":[{"tableIdentifier":"car_sales.csv","columnIdentifier":"column1"}]},"dependant":{"tableIdentifier":"car_sales.csv","columnIdentifier":"column15"}}

Car_id -> Phone
{"type":"FunctionalDependency","determinant":{"columnIdentifiers":[{"tableIdentifier":"car_sales.csv","columnIdentifier":"column1"}]},"dependant":{"tableIdentifier":"car_sales.csv","columnIdentifier":"column3"}}

Car_id -> Customer Name
{"type":"FunctionalDependency","determinant":{"columnIdentifiers":[{"tableIdentifier":"car_sales.csv","columnIdentifier":"column1"}]},"dependant":{"tableIdentifier":"car_sales.csv","columnIdentifier":"column5"}}

Car_id -> Annual Income
{"type":"FunctionalDependency","determinant":{"columnIdentifiers":[{"tableIdentifier":"car_sales.csv","columnIdentifier":"column1"}]},"dependant":{"tableIdentifier":"car_sales.csv","columnIdentifier":"column12"}}

Car_id -> Price ($)
{"type":"FunctionalDependency","determinant":{"columnIdentifie

In [26]:
find_fds_for_rhs('', newData, sorted = True, lhs_validator = lambda x : 'phone' in x )

[]

In [27]:
def sortnewData(dat, exclude_lhs):
    results = []
    for t in dat:
        if not any([ any([ exclude in l.lower() for exclude in exclude_lhs]) for l in t[0]]):
            results.append(t)
    results.sort(key=lambda x: len(x[0]))
    return results

In [28]:
for t in sortnewData(newData, ['phone', 'car_id']):
    if len(t[0] )< 5:
        print(t)

(['Dealer_Name'], 'Dealer_No ')
(['Engine'], 'Transmission')
(['Transmission'], 'Engine')
(['Price ($)', 'Model'], 'Company')
(['Date', 'Customer Name', 'Model'], 'Company')
(['Date', 'Customer Name', 'Model'], 'Body Style')
(['Customer Name', 'Dealer_Name', 'Model'], 'Company')
(['Dealer_No ', 'Customer Name', 'Model'], 'Company')
(['Dealer_Region', 'Customer Name', 'Model'], 'Company')
(['Body Style', 'Customer Name', 'Model'], 'Company')
(['Date', 'Dealer_Name', 'Model'], 'Company')
(['Dealer_Region', 'Date', 'Model'], 'Company')
(['Price ($)', 'Date', 'Customer Name', 'Annual Income'], 'Car_id')
(['Price ($)', 'Date', 'Customer Name', 'Annual Income'], 'Phone')
(['Price ($)', 'Date', 'Customer Name', 'Annual Income'], 'Model')
(['Price ($)', 'Date', 'Customer Name', 'Annual Income'], 'Company')
(['Price ($)', 'Date', 'Customer Name', 'Annual Income'], 'Dealer_Name')
(['Price ($)', 'Date', 'Customer Name', 'Annual Income'], 'Dealer_No ')
(['Price ($)', 'Date', 'Customer Name', 'Annu

In [29]:
def export_fds_to_csv(filename, subset):
    if not os.path.exists(filename):
        with open(filename, 'w', newline='') as f:
            writer = csv.writer(f)
            write_back = [['LHS', 'RHS']]
            write_back.extend(subset)
            writer.writerows(write_back)

In [30]:
subset = [t for t in sortnewData(newData, ['phone', 'car_id']) if len(t[0]) < 5]

In [31]:
filename = 'cars_relevant_fds'

export_fds_to_csv(filename, subset)

In [32]:
find_fds_for_rhs('Car_id', newData, sorted=True)

[(['Phone', 'Customer Name'], 'Car_id'),
 (['Price ($)', 'Phone'], 'Car_id'),
 (['Phone', 'Date'], 'Car_id'),
 (['Phone', 'Annual Income', 'Model'], 'Car_id'),
 (['Phone', 'Annual Income', 'Dealer_Name'], 'Car_id'),
 (['Dealer_No ', 'Phone', 'Annual Income'], 'Car_id'),
 (['Phone', 'Dealer_Region', 'Annual Income'], 'Car_id'),
 (['Phone', 'Dealer_Name', 'Model'], 'Car_id'),
 (['Dealer_No ', 'Phone', 'Model'], 'Car_id'),
 (['Phone', 'Dealer_Region', 'Model'], 'Car_id'),
 (['Phone', 'Gender', 'Model'], 'Car_id'),
 (['Phone', 'Dealer_Name', 'Company'], 'Car_id'),
 (['Dealer_No ', 'Phone', 'Company'], 'Car_id'),
 (['Phone', 'Dealer_Region', 'Company'], 'Car_id'),
 (['Body Style', 'Phone', 'Annual Income', 'Company'], 'Car_id'),
 (['Color', 'Phone', 'Annual Income', 'Company'], 'Car_id'),
 (['Color', 'Body Style', 'Phone', 'Annual Income'], 'Car_id'),
 (['Color', 'Phone', 'Annual Income', 'Engine'], 'Car_id'),
 (['Transmission', 'Color', 'Phone', 'Annual Income'], 'Car_id'),
 (['Body Style'

In [33]:
def readin_raw(path = '', head_string = '', seperator_head_string =',' ):
    labels = head_string.split(seperator_head_string)
    label_lookup = listToLabel(labels)
    print(label_lookup)
    data = []
    with open(path, 'r') as file:
        for line in file:
            #print(line)
            lhs, rhs = lineToDataStruct(line.strip().split(","), label_lookup)
            if any(['Year' in label for label in lhs]):
                print(line)
            #prettyPrint = lambda lst: ', '.join(lst)
            #print(prettyPrint(lhs), '->',rhs)
            data.append( (lhs, rhs) )
    return data

In [34]:
approx_airline_fds = readin_raw(path = 'results/2025-11-24_09-04-48_fds', head_string='Year,Quarter,Month,DayofMonth,DayOfWeek,FlightDate,UniqueCarrier,AirlineID,Carrier,TailNum,FlightNum,OriginAirportID,OriginAirportSeqID,OriginCityMarketID,Origin,OriginCityName,OriginState,OriginStateFips,OriginStateName,OriginWac,DestAirportID,DestAirportSeqID,DestCityMarketID,Dest,DestCityName,DestState,DestStateFips,DestStateName,DestWac,CRSDepTime,DepTime,DepDelay,DepDelayMinutes,DepDel15,DepartureDelayGroups,DepTimeBlk,TaxiOut,TaxiIn,CRSArrTime,ArrTime,ArrDelay,ArrDelayMinutes,ArrDel15,ArrivalDelayGroups,ArrTimeBlk,Cancelled,CancellationCode,Diverted,CRSElapsedTime,ActualElapsedTime,AirTime,Flights,Distance,DistanceGroup,CarrierDelay,WeatherDelay,NASDelay,SecurityDelay,LateAircraftDelay,FirstDepTime,TotalAddGTime,LongestAddGTime', seperator_head_string=',')

{'column1': 'Year', 'column2': 'Quarter', 'column3': 'Month', 'column4': 'DayofMonth', 'column5': 'DayOfWeek', 'column6': 'FlightDate', 'column7': 'UniqueCarrier', 'column8': 'AirlineID', 'column9': 'Carrier', 'column10': 'TailNum', 'column11': 'FlightNum', 'column12': 'OriginAirportID', 'column13': 'OriginAirportSeqID', 'column14': 'OriginCityMarketID', 'column15': 'Origin', 'column16': 'OriginCityName', 'column17': 'OriginState', 'column18': 'OriginStateFips', 'column19': 'OriginStateName', 'column20': 'OriginWac', 'column21': 'DestAirportID', 'column22': 'DestAirportSeqID', 'column23': 'DestCityMarketID', 'column24': 'Dest', 'column25': 'DestCityName', 'column26': 'DestState', 'column27': 'DestStateFips', 'column28': 'DestStateName', 'column29': 'DestWac', 'column30': 'CRSDepTime', 'column31': 'DepTime', 'column32': 'DepDelay', 'column33': 'DepDelayMinutes', 'column34': 'DepDel15', 'column35': 'DepartureDelayGroups', 'column36': 'DepTimeBlk', 'column37': 'TaxiOut', 'column38': 'Taxi

In [35]:
approx_airline_fds[:10]

[(['DepTime'], 'CarrierDelay'),
 (['OriginAirportID'], 'OriginStateName'),
 ([], 'UniqueCarrier'),
 (['FlightDate'], 'DayofMonth'),
 (['OriginAirportID'], 'OriginStateFips'),
 (['ArrTime'], 'TaxiIn'),
 (['OriginAirportSeqID'], 'OriginAirportID'),
 ([], 'Year'),
 (['ArrTime'], 'TailNum'),
 (['DestAirportID'], 'DestWac')]

In [36]:
arr_delay_sample = [ t for t in find_fds_for_rhs('ArrDel15', approx_airline_fds, sorted=True) if len(t[0]) > 0 and len(t[0]) < 6]

In [37]:
def fuzzy_filter(data, exclude_list = [], include_list = []):
    excludes_promise = lambda lhs: (not exclude_list) or (not any([any([ex.lower() in label.lower() for ex in exclude_list]) for label in lhs]))
    include_promise = lambda lhs: (not include_list) or (all([any(inc.lower() in label.lower() for label in lhs) for inc in include_list])) 
    return [t for t in data if excludes_promise(t[0]) and include_promise(t[0])]


In [38]:
genuine_fds_approx = []
fuzzy_filter(arr_delay_sample, exclude_list=['origin', 'dest', 'day', 'date','group'])
for t in fuzzy_filter(arr_delay_sample, exclude_list=['origin', 'dest', 'day', 'date','group'])[:2]: genuine_fds_approx.append(t) 

In [39]:
genuine_fds_approx

[(['ArrDelay'], 'ArrDel15'), (['ArrDelayMinutes'], 'ArrDel15')]

In [40]:
dep_delay_sample = [ t for t in find_fds_for_rhs('DepDel15', approx_airline_fds, sorted=True) if len(t[0]) > 0 and len(t[0]) < 6]

In [41]:
for t in dep_delay_sample[:3]: genuine_fds_approx.append(t)

In [42]:
for t in find_fds_for_rhs('OriginStateName', approx_airline_fds, sorted=True)[:8]: genuine_fds_approx.append(t)

In [43]:
genuine_fds_approx

[(['ArrDelay'], 'ArrDel15'),
 (['ArrDelayMinutes'], 'ArrDel15'),
 (['DepDelay'], 'DepDel15'),
 (['DepDelayMinutes'], 'DepDel15'),
 (['DepartureDelayGroups'], 'DepDel15'),
 (['OriginAirportID'], 'OriginStateName'),
 (['OriginAirportSeqID'], 'OriginStateName'),
 (['Origin'], 'OriginStateName'),
 (['OriginCityName'], 'OriginStateName'),
 (['OriginState'], 'OriginStateName'),
 (['OriginStateFips'], 'OriginStateName'),
 (['OriginWac'], 'OriginStateName'),
 (['OriginCityMarketID'], 'OriginStateName')]

In [44]:
approx_airline_resampled_fds = readin_raw(path = 'results/2025-11-24_10-56-29_fds', head_string='Year,Quarter,Month,DayofMonth,DayOfWeek,FlightDate,UniqueCarrier,AirlineID,Carrier,TailNum,FlightNum,OriginAirportID,OriginAirportSeqID,OriginCityMarketID,Origin,OriginCityName,OriginState,OriginStateFips,OriginStateName,OriginWac,DestAirportID,DestAirportSeqID,DestCityMarketID,Dest,DestCityName,DestState,DestStateFips,DestStateName,DestWac,CRSDepTime,DepTime,DepDelay,DepDelayMinutes,DepDel15,DepartureDelayGroups,DepTimeBlk,TaxiOut,TaxiIn,CRSArrTime,ArrTime,ArrDelay,ArrDelayMinutes,ArrDel15,ArrivalDelayGroups,ArrTimeBlk,Cancelled,CancellationCode,Diverted,CRSElapsedTime,ActualElapsedTime,AirTime,Flights,Distance,DistanceGroup,CarrierDelay,WeatherDelay,NASDelay,SecurityDelay,LateAircraftDelay,FirstDepTime,TotalAddGTime,LongestAddGTime', seperator_head_string=',')

{'column1': 'Year', 'column2': 'Quarter', 'column3': 'Month', 'column4': 'DayofMonth', 'column5': 'DayOfWeek', 'column6': 'FlightDate', 'column7': 'UniqueCarrier', 'column8': 'AirlineID', 'column9': 'Carrier', 'column10': 'TailNum', 'column11': 'FlightNum', 'column12': 'OriginAirportID', 'column13': 'OriginAirportSeqID', 'column14': 'OriginCityMarketID', 'column15': 'Origin', 'column16': 'OriginCityName', 'column17': 'OriginState', 'column18': 'OriginStateFips', 'column19': 'OriginStateName', 'column20': 'OriginWac', 'column21': 'DestAirportID', 'column22': 'DestAirportSeqID', 'column23': 'DestCityMarketID', 'column24': 'Dest', 'column25': 'DestCityName', 'column26': 'DestState', 'column27': 'DestStateFips', 'column28': 'DestStateName', 'column29': 'DestWac', 'column30': 'CRSDepTime', 'column31': 'DepTime', 'column32': 'DepDelay', 'column33': 'DepDelayMinutes', 'column34': 'DepDel15', 'column35': 'DepartureDelayGroups', 'column36': 'DepTimeBlk', 'column37': 'TaxiOut', 'column38': 'Taxi

In [45]:
approx_airline_resampled_fds[:10]

[([], 'Year'),
 (['TailNum'], 'TaxiIn'),
 (['FlightDate'], 'DayofMonth'),
 (['OriginAirportID'], 'OriginStateName'),
 (['OriginAirportID'], 'OriginStateFips'),
 (['AirlineID'], 'UniqueCarrier'),
 (['ArrDelayMinutes'], 'CarrierDelay'),
 (['OriginAirportSeqID'], 'OriginAirportID'),
 (['OriginAirportSeqID'], 'OriginStateFips'),
 (['Origin'], 'OriginStateFips')]

In [46]:
flight_date_sample_resample = [t for t in find_fds_for_rhs('FlightDate', approx_airline_resampled_fds, sorted=True) if len(t[0]) > 1]

In [47]:
fuzzy_filter(flight_date_sample_resample, exclude_list=['origin', 'dest','delaY', 'arr', 'taxi'])

[(['DepTimeBlk', 'AirlineID'], 'FlightDate'),
 (['DayOfWeek', 'AirlineID'], 'FlightDate'),
 (['DepTimeBlk', 'DayOfWeek'], 'FlightDate'),
 (['DepTimeBlk', 'DistanceGroup'], 'FlightDate'),
 (['DepDel15', 'DayOfWeek', 'DistanceGroup'], 'FlightDate'),
 (['DepDel15', 'DistanceGroup', 'AirlineID'], 'FlightDate')]

In [48]:
flight_date_sample= [t for t in find_fds_for_rhs('FlightDate', approx_airline_fds, sorted=True) if len(t[0]) > 1]

In [49]:
fuzzy_filter(flight_date_sample, exclude_list=['origin', 'dest','delaY', 'arr', 'taxi'])

[(['DepTimeBlk', 'Distance'], 'FlightDate'),
 (['DepTimeBlk', 'DistanceGroup'], 'FlightDate'),
 (['DayOfWeek', 'Distance'], 'FlightDate'),
 (['DepTimeBlk', 'DayOfWeek'], 'FlightDate'),
 (['DepDel15', 'DayOfWeek', 'DistanceGroup'], 'FlightDate')]

In [50]:
flight_date_sample_nonapprox = [t for t in find_fds_for_rhs('FlightDate', data, sorted=True) if len(t[0]) > 1]

In [51]:
fuzzy_filter(flight_date_sample, exclude_list=['dest', 'origin', 'delay'],include_list=['DayOfWeek'])

[(['DayOfWeek', 'Distance'], 'FlightDate'),
 (['TaxiOut', 'DayOfWeek'], 'FlightDate'),
 (['ArrTimeBlk', 'DayOfWeek'], 'FlightDate'),
 (['DepTimeBlk', 'DayOfWeek'], 'FlightDate'),
 (['TaxiIn', 'DayOfWeek'], 'FlightDate'),
 (['ArrDel15', 'DayOfWeek', 'DistanceGroup'], 'FlightDate'),
 (['DepDel15', 'DayOfWeek', 'DistanceGroup'], 'FlightDate')]

In [52]:
# AirlineID -> UniqueCarrier only holds for resampled 
genuine_fds_approx_resampled = [fuzzy_filter(find_fds_for_rhs('AirlineID', approx_airline_resampled_fds, sorted=True))[0]]

In [53]:
# tailnum
genuine_fds_approx.append(fuzzy_filter(find_fds_for_rhs('TailNum', approx_airline_fds, sorted=True), exclude_list=['time', 'day', 'taxi','delay'])[0])

In [54]:
#flightnum 
fuzzy_filter(find_fds_for_rhs('FlightNum', approx_airline_fds, sorted=True), exclude_list=['Distance', 'taxi', 'delay'], include_list=['FlightDate'])

[(['ArrTimeBlk', 'FlightDate'], 'FlightNum'),
 (['DepTimeBlk', 'FlightDate'], 'FlightNum'),
 (['OriginAirportSeqID', 'FlightDate'], 'FlightNum'),
 (['OriginAirportID', 'FlightDate'], 'FlightNum'),
 (['Origin', 'FlightDate'], 'FlightNum'),
 (['Dest', 'FlightDate'], 'FlightNum'),
 (['DestAirportID', 'FlightDate'], 'FlightNum'),
 (['DestAirportSeqID', 'FlightDate'], 'FlightNum'),
 (['OriginCityName', 'FlightDate'], 'FlightNum'),
 (['DestCityName', 'FlightDate'], 'FlightNum'),
 (['OriginCityMarketID', 'FlightDate'], 'FlightNum'),
 (['DestCityMarketID', 'FlightDate'], 'FlightNum'),
 (['OriginState', 'FlightDate'], 'FlightNum'),
 (['OriginWac', 'FlightDate'], 'FlightNum'),
 (['OriginStateFips', 'FlightDate'], 'FlightNum'),
 (['OriginStateName', 'FlightDate'], 'FlightNum'),
 (['DestStateName', 'FlightDate'], 'FlightNum'),
 (['DestState', 'FlightDate'], 'FlightNum'),
 (['DestStateFips', 'FlightDate'], 'FlightNum'),
 (['DestWac', 'FlightDate'], 'FlightNum')]

In [55]:
#OriginStateFips 
genuine_fds_approx.append(fuzzy_filter(find_fds_for_rhs('OriginStateFips', approx_airline_fds, sorted=True), include_list=['OriginState'])[0])

In [56]:
#depDelay - unfortunately not possible 
fuzzy_filter(find_fds_for_rhs('DepDelay', approx_airline_fds, sorted=True))

[(['DepTime'], 'DepDelay'),
 (['ArrTime'], 'DepDelay'),
 (['TailNum'], 'DepDelay'),
 (['ActualElapsedTime'], 'DepDelay'),
 (['CRSArrTime'], 'DepDelay'),
 (['AirTime'], 'DepDelay'),
 (['FlightNum'], 'DepDelay'),
 (['CRSElapsedTime'], 'DepDelay'),
 (['CRSDepTime'], 'DepDelay'),
 (['ArrDelay', 'Distance'], 'DepDelay'),
 (['DayofMonth', 'ArrDelay'], 'DepDelay'),
 (['DayofMonth', 'Distance'], 'DepDelay'),
 (['ArrDelay', 'FlightDate'], 'DepDelay'),
 (['Distance', 'FlightDate'], 'DepDelay'),
 (['TaxiOut', 'FlightDate'], 'DepDelay'),
 (['TaxiOut', 'DayofMonth'], 'DepDelay'),
 (['TaxiOut', 'Distance'], 'DepDelay'),
 (['TaxiOut', 'ArrDelay'], 'DepDelay'),
 (['ArrDelay', 'ArrTimeBlk'], 'DepDelay'),
 (['DayofMonth', 'ArrTimeBlk'], 'DepDelay'),
 (['ArrTimeBlk', 'FlightDate'], 'DepDelay'),
 (['TaxiOut', 'ArrTimeBlk'], 'DepDelay'),
 (['ArrTimeBlk', 'Distance'], 'DepDelay'),
 (['DepTimeBlk', 'ArrDelay'], 'DepDelay'),
 (['DepTimeBlk', 'FlightDate'], 'DepDelay'),
 (['DepTimeBlk', 'DayofMonth'], 'DepDela

In [57]:
#DepartureDelayGroups
genuine_fds_approx.append(fuzzy_filter(find_fds_for_rhs('DepartureDelayGroups', approx_airline_fds, sorted=True))[0])

In [58]:
genuine_fds_approx.append(fuzzy_filter(find_fds_for_rhs('DepTimeBlk', approx_airline_fds, sorted=True, lhs_size_max=1), include_list=['DepTime'])[1])

In [59]:
genuine_fds_approx

[(['ArrDelay'], 'ArrDel15'),
 (['ArrDelayMinutes'], 'ArrDel15'),
 (['DepDelay'], 'DepDel15'),
 (['DepDelayMinutes'], 'DepDel15'),
 (['DepartureDelayGroups'], 'DepDel15'),
 (['OriginAirportID'], 'OriginStateName'),
 (['OriginAirportSeqID'], 'OriginStateName'),
 (['Origin'], 'OriginStateName'),
 (['OriginCityName'], 'OriginStateName'),
 (['OriginState'], 'OriginStateName'),
 (['OriginStateFips'], 'OriginStateName'),
 (['OriginWac'], 'OriginStateName'),
 (['OriginCityMarketID'], 'OriginStateName'),
 (['FlightNum'], 'TailNum'),
 (['OriginState'], 'OriginStateFips'),
 (['DepDelay'], 'DepartureDelayGroups'),
 (['DepTime'], 'DepTimeBlk')]

In [62]:
genuine_fds_approx.append ( fuzzy_filter(find_fds_for_rhs('Cancelled', approx_airline_fds, sorted=True), include_list=['CancellationCode'])[0])

In [81]:
approx_airline_fds_grouped = readin_raw(path = 'results/2025-11-24_18-28-09_fds', head_string='Year,Quarter,Month,DayofMonth,DayOfWeek,FlightDate,UniqueCarrier,AirlineID,Carrier,TailNum,FlightNum,OriginAirportID,OriginAirportSeqID,OriginCityMarketID,Origin,OriginCityName,OriginState,OriginStateFips,OriginStateName,OriginWac,DestAirportID,DestAirportSeqID,DestCityMarketID,Dest,DestCityName,DestState,DestStateFips,DestStateName,DestWac,CRSDepTime,DepTime,DepDelay,DepDelayMinutes,DepDel15,DepartureDelayGroups,DepTimeBlk,TaxiOut,TaxiIn,CRSArrTime,ArrTime,ArrDelay,ArrDelayMinutes,ArrDel15,ArrivalDelayGroups,ArrTimeBlk,Cancelled,CancellationCode,Diverted,CRSElapsedTime,ActualElapsedTime,AirTime,Flights,Distance,DistanceGroup,CarrierDelay,WeatherDelay,NASDelay,SecurityDelay,LateAircraftDelay,FirstDepTime,TotalAddGTime,LongestAddGTime', seperator_head_string=',')

{'column1': 'Year', 'column2': 'Quarter', 'column3': 'Month', 'column4': 'DayofMonth', 'column5': 'DayOfWeek', 'column6': 'FlightDate', 'column7': 'UniqueCarrier', 'column8': 'AirlineID', 'column9': 'Carrier', 'column10': 'TailNum', 'column11': 'FlightNum', 'column12': 'OriginAirportID', 'column13': 'OriginAirportSeqID', 'column14': 'OriginCityMarketID', 'column15': 'Origin', 'column16': 'OriginCityName', 'column17': 'OriginState', 'column18': 'OriginStateFips', 'column19': 'OriginStateName', 'column20': 'OriginWac', 'column21': 'DestAirportID', 'column22': 'DestAirportSeqID', 'column23': 'DestCityMarketID', 'column24': 'Dest', 'column25': 'DestCityName', 'column26': 'DestState', 'column27': 'DestStateFips', 'column28': 'DestStateName', 'column29': 'DestWac', 'column30': 'CRSDepTime', 'column31': 'DepTime', 'column32': 'DepDelay', 'column33': 'DepDelayMinutes', 'column34': 'DepDel15', 'column35': 'DepartureDelayGroups', 'column36': 'DepTimeBlk', 'column37': 'TaxiOut', 'column38': 'Taxi

In [83]:
fuzzy_filter(find_fds_for_rhs('ArrTime', approx_airline_fds_grouped, sorted=True), include_list=['DepTime'])

[(['DepTime'], 'ArrTime'),
 (['CRSDepTime'], 'ArrTime'),
 (['OriginCityName', 'DepTimeBlk'], 'ArrTime'),
 (['OriginAirportID', 'DepTimeBlk'], 'ArrTime'),
 (['Origin', 'DepTimeBlk'], 'ArrTime'),
 (['DestAirportID', 'DepTimeBlk'], 'ArrTime'),
 (['DestAirportSeqID', 'DepTimeBlk'], 'ArrTime'),
 (['DestCityName', 'DepTimeBlk'], 'ArrTime'),
 (['OriginAirportSeqID', 'DepTimeBlk'], 'ArrTime'),
 (['OriginCityMarketID', 'DepTimeBlk'], 'ArrTime'),
 (['DestCityMarketID', 'DepTimeBlk'], 'ArrTime'),
 (['Dest', 'DepTimeBlk'], 'ArrTime'),
 (['DepTimeBlk', 'ArrDelay'], 'ArrTime'),
 (['DepTimeBlk', 'FlightDate'], 'ArrTime'),
 (['DepDelay', 'DepTimeBlk'], 'ArrTime'),
 (['DepTimeBlk', 'DayofMonth'], 'ArrTime'),
 (['DepTimeBlk', 'TaxiOut'], 'ArrTime'),
 (['DepTimeBlk', 'TaxiIn'], 'ArrTime'),
 (['OriginStateName', 'DepTimeBlk'], 'ArrTime'),
 (['OriginState', 'DepTimeBlk'], 'ArrTime'),
 (['OriginWac', 'DepTimeBlk'], 'ArrTime'),
 (['OriginStateFips', 'DepTimeBlk'], 'ArrTime'),
 (['DestStateName', 'DepTimeBlk'

In [84]:
genuine_fds

[(['CRSArrTime', 'ArrTime'], 'ArrDelay')]

In [85]:
genuine_fds_approx

[(['ArrDelay'], 'ArrDel15'),
 (['ArrDelayMinutes'], 'ArrDel15'),
 (['DepDelay'], 'DepDel15'),
 (['DepDelayMinutes'], 'DepDel15'),
 (['DepartureDelayGroups'], 'DepDel15'),
 (['OriginAirportID'], 'OriginStateName'),
 (['OriginAirportSeqID'], 'OriginStateName'),
 (['Origin'], 'OriginStateName'),
 (['OriginCityName'], 'OriginStateName'),
 (['OriginState'], 'OriginStateName'),
 (['OriginStateFips'], 'OriginStateName'),
 (['OriginWac'], 'OriginStateName'),
 (['OriginCityMarketID'], 'OriginStateName'),
 (['FlightNum'], 'TailNum'),
 (['OriginState'], 'OriginStateFips'),
 (['DepDelay'], 'DepartureDelayGroups'),
 (['DepTime'], 'DepTimeBlk'),
 (['CancellationCode'], 'Cancelled')]